In [ ]:
import os
import json
from PIL import Image
from pdf2image import convert_from_path
from datasets import Dataset
from sklearn.model_selection import train_test_split
import torch

from transformers import (
    DonutProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator
)


In [ ]:
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
#VRAM 비우기+메모리 조각화 방지

In [ ]:

# 1. 경로 설정
data_dir = r"data/dataset"
model_name = "naver-clova-ix/donut-base"

def convert_pdf_to_png(pdf_path, output_path):
    # 첫 페이지만 이미지로 변환
    pages = convert_from_path(pdf_path, dpi=300, first_page=1, last_page=1)
    if pages:
        pages[0].save(output_path, "PNG")

# 2. 데이터 로딩 및 나누기
data = []
for json_file in os.listdir(data_dir):
    if json_file.endswith(".json"):
        base = json_file.replace(".json", "")
        pdf_path = os.path.join(data_dir, base + ".pdf")
        image_path = os.path.join(data_dir, base + ".png")

        # PDF → PNG 변환 (존재하지 않을 때만)
        if os.path.exists(pdf_path) and not os.path.exists(image_path):
            convert_pdf_to_png(pdf_path, image_path)

        # 이미지가 존재하면 데이터셋에 추가
        if os.path.exists(image_path):
            with open(os.path.join(data_dir, json_file), "r", encoding="utf-8") as f:
                label = json.load(f)
            data.append({
                "image_path": image_path,
                "ground_truth": label["ground_truth"]["gt_parse"]
            })

train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

# 3. 모델과 Processor

# 3-1 
#model_path = "outputs/donut_finetuned"
#model = VisionEncoderDecoderModel.from_pretrained(model_path)
#processor = DonutProcessor.from_pretrained(model_path)

processor = DonutProcessor.from_pretrained(model_name)
model = VisionEncoderDecoderModel.from_pretrained(model_name)
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids("<s_gt_parse>")#디코더의 처음 시작 토큰 정의 
model.config.eos_token_id = processor.tokenizer.eos_token_id#End of Sequence 토큰 -> 예측이 끝날때 Decorder가 이 토큰을 만나면 멈춤
#안멈추면 계속 쓸데없는 예측을 함
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.use_cache = False
model.gradient_checkpointing_enable() #중간 계산값을 저장하지 않고, Backward시 재계산 / VRAM 20%정도 절약 가능



print("decoder_start_token_id:", model.config.decoder_start_token_id)
print("eos_token_id:", processor.tokenizer.eos_token_id)  



In [ ]:
# 4. 전처리 함수
def json_to_token_string(json_obj):
    # Donut은 <s_gt_parse> ... </s> 구조를 예측함
    return "<s_gt_parse>" + json.dumps(json_obj, ensure_ascii=False) + "</s>"


def preprocess(example):
    image_path = example["image_path"]
    image = Image.open(image_path).convert("RGB")
    task_prompt = "<s_gt_parse>"

    # 이미지 전처리는 processor.image_processor로 명시적으로 처리
    image_encoding = processor.image_processor(
        images=image,
        return_tensors="pt",
        size={"height": 720, "width": 960}
    )

    # 텍스트 토크나이즈 (prompt로)
    text_encoding = processor.tokenizer(
        task_prompt,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=512
    )

    # 정답 텍스트를 JSON → 문자열로 변환
    target_text = json_to_token_string(example["ground_truth"])

    # 라벨 토큰화
    labels = processor.tokenizer(
        target_text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=512
    ).input_ids[0]

    labels[labels == processor.tokenizer.pad_token_id] = -100

    return {
        "pixel_values": image_encoding["pixel_values"][0],
        "labels": labels
    }
# 5. 전처리 적용
train_dataset = train_dataset.map(preprocess)
val_dataset = val_dataset.map(preprocess)
#데이터셋에 이미지 path 등 나머지 날리기 ->오류 방지
train_dataset = train_dataset.remove_columns(["image_path", "ground_truth"])
val_dataset = val_dataset.remove_columns(["image_path", "ground_truth"])

# 6. 학습 설정
training_args = Seq2SeqTrainingArguments(
    output_dir="outputs/donut_finetuned",
    per_device_train_batch_size=1, #train 배치크기
    per_device_eval_batch_size=1, #eval 배치크기
    learning_rate=5e-5, #학습률
    weight_decay=0.01, #가중치 감쇠(과적합 방지)
    warmup_ratio=0.05, #학습 초반에 LR을 천천히 증가시켜 안정화
    max_grad_norm=1.0, #그래디언트(기울기) 폭발 방지
    num_train_epochs=15, #에폭 수
    logging_dir="./logs",
    logging_steps=10,
    evaluation_strategy="epoch", #학습 중 평가함
    save_strategy="epoch", #매 에폭 저장
    predict_with_generate=True, #예측결과 디코딩을 위해 필수
    remove_unused_columns=False, #반드시 false
    fp16=True #True면 좋지만 GPU용 PyTorch깔고 GPU로 학습시켜야하는데 그게 안되서 지금 CPU
)

# 7. Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=processor.tokenizer,
    data_collator=default_data_collator
)

# 8. 학습 실행
trainer.train()

# 9. 모델 저장
model.save_pretrained("outputs/donut_finetuned") 
#model.safetensors가 '학습된 가중치'가 들어있는, 핵심적인 딥러닝 모델 파라미터
#config.json 모델 구조 설정: 예) hidden_size, decoder_start_token_id, pad_token_id 등 재로딩 시 정확히 동일한 구조로 모델을 불러오기 위한 참조용
#preprocessor_config.json DonutProcessor 전용 설정: 이미지 입력 사이즈, normalize 방법, tokenizer 이름 등 포함됨
#tokenizer_config.json, vocab.json, merges.txt tokenizer의 동작 방식, 사용되는 BPE 병합 규칙 및 단어 사전 정보

processor.save_pretrained("outputs/donut_finetuned")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# train loss 및 val loss 따로 필터링
df_loss = df_log[df_log["loss"].notna()]
df_val = df_log[df_log["eval_loss"].notna()]

# 손실 그래프 그리기
plt.figure(figsize=(10, 5))
plt.plot(df_loss["step"], df_loss["loss"], label="train_loss", marker='o')
plt.plot(df_val["step"], df_val["eval_loss"], label="val_loss", marker='x')
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
print(torch.cuda.is_available())

In [ ]:
print(type(train_dataset[0]["pixel_values"]))

In [ ]:
import accelerate
print(accelerate.__version__)  # 최소 0.22.0 이상이면 안전

In [ ]:
print(torch.cuda.is_available())